# TISER Dataset Chunker for Manual Translation

This notebook is designed to support a **human-in-the-loop translation process**
for the TISER dataset.

The goal is to:
- Load a large JSON dataset (train or test)
- Split it into manageable chunks
- Display **one chunk at a time**
- Allow manual translation using ChatGPT / Gemini
- Avoid confusion or loss of alignment between chunks

Each execution of the main cell will output **exactly one chunk**.

## Configuration

Set:
- the input dataset path
- the chunk size
- the progress file used to remember which chunk was last shown

The progress file ensures that you can safely stop and resume the process
without reprocessing the same examples.

In [6]:
import json
from pathlib import Path

# ===== USER CONFIG =====
DATASET_PATH = Path("/Users/usermastro/Desktop/Primo_Semestre_2526/DNLP/Project/TISER_repo/tools/datasets/TISER_test_10pct.json")   # change to train or test
CHUNK_SIZE = 20                                    # examples per chunk
PROGRESS_FILE = Path(".translation_progress.json") # internal state
test = True  # set to False for training, True for testing
# =======================

## Load Dataset

The dataset is expected to be a JSON list where each element corresponds
to one TISER example.

No modification is applied at this stage.

In [2]:
with DATASET_PATH.open("r", encoding="utf-8") as f:
    dataset = []
    with DATASET_PATH.open("r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                dataset.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {line_num}: {e}") from e

total_examples = len(dataset)
total_chunks = (total_examples + CHUNK_SIZE - 1) // CHUNK_SIZE

print(f"Loaded dataset: {DATASET_PATH}")
print(f"Total examples: {total_examples}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Total chunks: {total_chunks}")

Loaded dataset: /Users/usermastro/Desktop/Primo_Semestre_2526/DNLP/Project/TISER_repo/tools/datasets/TISER_test_10pct.json
Total examples: 386
Chunk size: 20
Total chunks: 20


## One-time: store the translated TISER prompt

All samples share the same `prompt`, so we translate it once and reuse it.

Create a file (or paste in a cell) that contains the **Italian translation** of the TISER prompt.
The pipeline will re-insert this prompt into every translated example during the append step.

In [3]:
from pathlib import Path

# Store the translated prompt here (one time)
PROMPT_IT_PATH = Path("test_TISER_prompt_it.txt")

if not PROMPT_IT_PATH.exists():
    raise FileNotFoundError(
        f"Missing {PROMPT_IT_PATH}. Create it and paste the translated prompt there."
    )

prompt_it = PROMPT_IT_PATH.read_text(encoding="utf-8").strip()
print("Loaded prompt_it. Characters:", len(prompt_it))

Loaded prompt_it. Characters: 1918


---

---

## Export Next Chunk for Human-in-the-Loop Translation

This cell performs the **core operational step** of the human-in-the-loop translation pipeline.

Specifically, each execution:

1. **Reads the current progress state** to determine which chunk should be processed next.
2. **Extracts the next chunk** of the dataset based on the configured chunk size.
3. **Removes the `prompt` field** from each example, since the TISER prompt is fixed and translated only once.
4. **Exports two artifacts**:
   - A **pure JSON file** (`chunk_xxxx.json`) containing only the dataset examples, which remains machine-readable and can be validated programmatically.
   - A **ChatGPT/Gemini-ready text file** (`chunk_xxxx_for_chat.txt`) that contains:
     - The full translation prompt (loaded from an external file),
     - Followed by the JSON chunk to be translated.

The text file is intentionally **not valid JSON**, as it embeds natural language instructions before the JSON payload.
This design allows the user to **copy and paste the entire file content directly into ChatGPT or Gemini**
without manual assembly of the prompt and the data.

After exporting the files, the cell **updates the progress file** so that subsequent executions automatically move to the next chunk.
This ensures a deterministic, resumable, and error-resistant translation workflow.

In [68]:
import json
import re
from pathlib import Path

# =======================
# PATHS
# =======================
OUT_DIR = Path("chunks_to_translate")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PROMPT_PATH = Path("prompt_for_CHAT.txt")
if not PROMPT_PATH.exists():
    raise FileNotFoundError(f"Prompt file not found: {PROMPT_PATH.resolve()}")

PROMPT_TEXT = PROMPT_PATH.read_text(encoding="utf-8").strip()

# =======================
# HELPERS — SAFE UNICODE FIX
# =======================

_unicode_pattern = re.compile(r'\\u([0-9a-fA-F]{4})')

def _decode_unicode_only(s: str) -> str:
    """
    Decode ONLY \\uXXXX sequences into real Unicode characters.
    Leaves \\n, \\t, \\\\ etc. untouched.
    """
    # Fix missing-backslash forms: u00e9 -> \u00e9
    s = re.sub(r'(?<!\\)\bu00([0-9a-fA-F]{2})\b', r'\\u00\1', s, flags=re.IGNORECASE)

    def _replace(match):
        codepoint = int(match.group(1), 16)
        return chr(codepoint)

    return _unicode_pattern.sub(_replace, s)

def make_chatgpt_txt(prompt_text: str, json_obj) -> str:
    """
    Build prompt + JSON for ChatGPT, decoding ONLY unicode letters.
    Keeps \\n as \\n.
    """
    json_text = json.dumps(json_obj, ensure_ascii=False, indent=2)
    full = prompt_text + "\n\n" + json_text
    return _decode_unicode_only(full)

# =======================
# LOAD PROGRESS
# =======================
if PROGRESS_FILE.exists():
    with PROGRESS_FILE.open("r", encoding="utf-8") as f:
        progress = json.load(f)
    current_chunk_idx = int(progress.get("current_chunk", 0))
else:
    current_chunk_idx = 0

print(f"Current chunk index: {current_chunk_idx}/{total_chunks}")

# =======================
# EXPORT NEXT CHUNK
# =======================
if current_chunk_idx >= total_chunks:
    print("All chunks have already been processed.")
else:
    start = current_chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, total_examples)
    chunk = dataset[start:end]

    # Remove "prompt" and "context"
    chunk_wo_prompt_wo_context = []
    for ex in chunk:
        ex_dict = dict(ex)
        ex_dict.pop("prompt", None)
        ex_dict.pop("context", None)
        chunk_wo_prompt_wo_context.append(ex_dict)

    chunk_number = current_chunk_idx + 1

    # ---- JSON BACKUP ----
    json_path = OUT_DIR / f"chunk_{chunk_number:04d}.json"
    json_path.write_text(
        json.dumps(chunk_wo_prompt_wo_context, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    # ---- TXT FOR LLM ----
    txt_path = OUT_DIR / f"chunk_{chunk_number:04d}_for_LLM.txt"
    chat_txt = make_chatgpt_txt(PROMPT_TEXT, chunk_wo_prompt_wo_context)
    txt_path.write_text(chat_txt, encoding="utf-8")

    # Update progress
    with PROGRESS_FILE.open("w", encoding="utf-8") as f:
        json.dump({"current_chunk": chunk_number}, f, ensure_ascii=False, indent=2)

    print(f"Saved JSON chunk: {json_path.resolve()}")
    print(f"Saved ChatGPT-ready file (\\n preserved): {txt_path.resolve()}")
    print(f"Next chunk will be: {chunk_number + 1:04d}")

Current chunk index: 20/20
All chunks have already been processed.


---

## Validate and Sanitize the Last Translated Chunk

This cell performs a **full validation and cleanup pass** on the most recently translated chunk,
automatically inferred from the progress file.

It is designed to catch and fix the most common issues that arise when JSON chunks are translated
manually using ChatGPT or Gemini.

Specifically, the cell executes the following steps:

### 1. Identify the Last Generated Chunk
- Reads the progress file to determine the last exported chunk index.
- Locates the corresponding file `chunks_to_translate/chunk_XXXX.json`.
- Fails immediately if the progress file or chunk file is missing.

### 2. Sanitize Common Copy/Paste Artifacts
- Replaces smart quotes and typographic characters with standard ASCII equivalents.
- Normalizes ellipses and other problematic Unicode punctuation.
- This step is necessary because LLM outputs often introduce characters that break JSON parsing.

### 3. Parse and Rewrite Valid JSON
- Attempts to parse the sanitized text using `json.loads`.
- If parsing fails, prints:
  - the exact JSON error,
  - line and column information,
  - a small surrounding context window to quickly identify the issue.
- If parsing succeeds, rewrites the file as clean, pretty-printed UTF-8 JSON.
  This guarantees the chunk is stable and machine-readable.

### 4. Reconstruct the English Reference Chunk
- Rebuilds the corresponding English chunk directly from the original dataset,
  using the same chunk indices.
- Removes the `prompt` field to match the translated chunk structure.
- This reconstructed chunk acts as a ground-truth reference for alignment checks.

### 5. Structural and Alignment Validation
The cell verifies that:
- The translated chunk is a JSON array.
- Every element is a JSON object.
- The number of examples matches between English and Italian.
- The `question_id` order is identical (no shuffling or missing samples).
- All required keys are present in each translated example:
  `dataset_name`, `question_id`, `question`, `answer`, `output`, `context`.

### 6. Final Outcome
- If all checks pass, the cell prints a success message.
- At this point, the chunk is guaranteed to be:
  - valid JSON,
  - structurally aligned with the original dataset,
  - safe to append to the final Italian train or test file.

In [66]:
import json
import re
from pathlib import Path

OUT_DIR = Path("chunks_to_translate")

# -----------------------------
# 0) Deduce last chunk from progress
# -----------------------------
if not PROGRESS_FILE.exists():
    raise FileNotFoundError(f"Progress file not found: {PROGRESS_FILE.resolve()}")

with PROGRESS_FILE.open("r", encoding="utf-8") as f:
    progress = json.load(f)

last_chunk_number = int(progress.get("current_chunk", 0))  # 1-based
if last_chunk_number <= 0:
    raise ValueError(
        f"progress['current_chunk'] is {last_chunk_number}. "
        "Run the export cell first to generate a chunk."
    )

chunk_path = OUT_DIR / f"chunk_{last_chunk_number:04d}.json"
if not chunk_path.exists():
    raise FileNotFoundError(f"Chunk file not found: {chunk_path.resolve()}")

print("Last chunk number:", last_chunk_number)
print("Chunk path:", chunk_path.resolve())

# -----------------------------
# 1) Read file + fix ONLY JSON-breaking delimiter characters pre-parse
# -----------------------------
text = chunk_path.read_text(encoding="utf-8")

text = (text
    .replace("“", "\"")
    .replace("”", "\"")
    .replace("„", "\"")
    .replace("«", "\"")
    .replace("»", "\"")
    .replace("‘", "'")
    .replace("’", "'")
    .replace("…", "...")
)

# -----------------------------
# 2) Parse JSON (hard fail with context if invalid)
# -----------------------------
try:
    chunk_it_wo_prompt = json.loads(text)
except json.JSONDecodeError as e:
    print("❌ Still invalid JSON after delimiter sanitization.")
    print("Error:", e)
    print("Line:", e.lineno, "Col:", e.colno, "Pos:", e.pos)

    lines = text.splitlines()
    start_ctx = max(e.lineno - 3, 0)
    end_ctx = min(e.lineno + 2, len(lines))

    print("\n--- Context ---")
    for i in range(start_ctx, end_ctx):
        marker = ">>" if (i + 1) == e.lineno else "  "
        print(f"{marker} {i+1:04d}: {lines[i]}")
    raise

# -----------------------------
# 3) Post-parse sanitization (apply everywhere EXCEPT question_id)
# -----------------------------
_u00_fix_re = re.compile(r'(?<!\\)u00([0-9a-fA-F]{2})')
_dots_fix_re = re.compile(r'\.\.+')

def _sanitize_string(s: str) -> str:
    s = (s
        .replace("“", "\"")
        .replace("”", "\"")
        .replace("„", "\"")
        .replace("«", "\"")
        .replace("»", "\"")
        .replace("‘", "'")
        .replace("’", "'")
        .replace("…", "...")
    )
    # Fix broken unicode-like sequences produced by the model: "Alaquu00e0s" -> "Alaqu\u00e0s"
    s = _u00_fix_re.sub(r'\\u00\1', s)
    # Collapse accidental multiple dots
    s = _dots_fix_re.sub('.', s)
    return s

def sanitize_obj(obj, parent_key=None):
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            if k == "question_id":
                out[k] = v  # do not sanitize here
            else:
                out[k] = sanitize_obj(v, parent_key=k)
        return out
    elif isinstance(obj, list):
        return [sanitize_obj(x, parent_key=parent_key) for x in obj]
    elif isinstance(obj, str):
        if parent_key == "question_id":
            return obj
        return _sanitize_string(obj)
    else:
        return obj

chunk_it_wo_prompt = sanitize_obj(chunk_it_wo_prompt)

# -----------------------------
# 4) Type checks for IT chunk
# -----------------------------
if not isinstance(chunk_it_wo_prompt, list):
    raise ValueError(f"{chunk_path.name} is not a JSON array.")
for i, item in enumerate(chunk_it_wo_prompt):
    if not isinstance(item, dict):
        raise ValueError(f"{chunk_path.name}: element {i} is not a JSON object.")

# -----------------------------
# 5) Reconstruct EN reference chunk (without prompt)
# -----------------------------
start = (last_chunk_number - 1) * CHUNK_SIZE
end = min(start + CHUNK_SIZE, total_examples)
chunk_en = dataset[start:end]

chunk_en_wo_prompt = []
for ex in chunk_en:
    ex_dict = dict(ex)
    ex_dict.pop("prompt", None)
    chunk_en_wo_prompt.append(ex_dict)

print("EN reference range:", start, "→", end - 1, "count =", len(chunk_en_wo_prompt))
print("IT translated count:", len(chunk_it_wo_prompt))

# -----------------------------
# 6) Validation: length
# -----------------------------
if len(chunk_it_wo_prompt) != len(chunk_en_wo_prompt):
    raise ValueError(
        f"Different number of examples: EN={len(chunk_en_wo_prompt)} IT={len(chunk_it_wo_prompt)}"
    )

# -----------------------------
# 7) HARD FIX: enforce EN question_id on IT objects (index-by-index)
#    This guarantees sanitization never creates a mismatch on IDs.
# -----------------------------
for i in range(len(chunk_it_wo_prompt)):
    chunk_it_wo_prompt[i]["question_id"] = chunk_en_wo_prompt[i].get("question_id")

# -----------------------------
# 8) Rewrite clean, stable JSON back to file (canonical UTF-8)
# -----------------------------
chunk_path.write_text(
    json.dumps(chunk_it_wo_prompt, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print("✅ Sanitized + rewritten valid JSON (question_id enforced from EN):", chunk_path.resolve())

# -----------------------------
# 9) Validation: IDs + required keys
# -----------------------------
ids_en = [ex.get("question_id") for ex in chunk_en_wo_prompt]
ids_it = [ex.get("question_id") for ex in chunk_it_wo_prompt]

if ids_en != ids_it:
    for i, (a, b) in enumerate(zip(ids_en, ids_it)):
        if a != b:
            raise ValueError(f"question_id mismatch at index {i}: EN={a} IT={b}")
    raise ValueError("question_id mismatch (order differs).")

if test:
    required_keys = {"dataset_name", "question_id", "question", "answer"}
else:
    required_keys = {"dataset_name", "question_id", "question", "answer", "output"}
for i, ex in enumerate(chunk_it_wo_prompt):
    missing = required_keys - set(ex.keys())
    if missing:
        raise ValueError(
            f"Missing keys at index {i} (question_id={ex.get('question_id')}): {missing}"
        )

print("✅ Validation passed.")
print("Examples:", len(chunk_it_wo_prompt))

Last chunk number: 20
Chunk path: /Users/usermastro/Desktop/Primo_Semestre_2526/DNLP/Project/TISER_repo/tools/translation/chunks_to_translate/chunk_0020.json
EN reference range: 380 → 385 count = 6
IT translated count: 6
✅ Sanitized + rewritten valid JSON (question_id enforced from EN): /Users/usermastro/Desktop/Primo_Semestre_2526/DNLP/Project/TISER_repo/tools/translation/chunks_to_translate/chunk_0020.json
✅ Validation passed.
Examples: 6


---

# Reinsert the translated prompt and append to the cumulative Italian dataset

We now:
- add `prompt: <prompt_it>` back into each example
- append them to a growing output file (JSONL is recommended)
- this creates a single Italian dataset file that grows chunk by chunk

Why JSONL:
- safe incremental append (no need to re-write a giant JSON array each time)
- robust against partial writes and large file sizes

In [67]:
from pathlib import Path
import json

OUT_IT_JSONL = Path("TISER_it.jsonl")  # change name if you want

# Flag: True for test, False for train
test = True

def build_full_prompt(base_prompt: str, question_val: str, context_val: str, test: bool) -> str:
    """
    - test=True: insert Question + Temporal context BEFORE '### Answer:' (last occurrence).
    - test=False: append Question + Temporal context at END of prompt.
    """
    if test:
        marker = "### Answer:"
        if marker in base_prompt:
            idx = base_prompt.rfind(marker)
            head = base_prompt[:idx].rstrip()
            tail = base_prompt[idx:]  # includes ### Answer:
            return (
                f"{head}\n\n"
                f"Question: {question_val}\n\n"
                f"Temporal context: {context_val}\n\n"
                f"{tail}"
            )
        else:
            return (
                f"{base_prompt.strip()}\n\n"
                f"Question: {question_val}\n\n"
                f"Temporal context: {context_val}\n\n"
                f"### Answer:"
            )
    else:
        return (
            f"{base_prompt.strip()}\n\n"
            f"Question: {question_val}\n\n"
            f"Temporal context: {context_val}"
        )

def insert_prompt_in_order(ex: dict, prompt_text: str, test: bool) -> dict:
    """
    - test=True: prompt after 'question' and before 'answer'
    - test=False: prompt after 'answer' and before 'output'
    """
    ex_clean = {k: v for k, v in ex.items() if k != "prompt"}

    new_ex = {}
    inserted = False

    if test:
        for k, v in ex_clean.items():
            new_ex[k] = v
            if k == "question" and not inserted:
                new_ex["prompt"] = prompt_text
                inserted = True

        if not inserted:
            new_ex = {}
            for k, v in ex_clean.items():
                if k == "answer" and not inserted:
                    new_ex["prompt"] = prompt_text
                    inserted = True
                new_ex[k] = v
    else:
        for k, v in ex_clean.items():
            new_ex[k] = v
            if k == "answer" and not inserted:
                new_ex["prompt"] = prompt_text
                inserted = True

        if not inserted:
            new_ex = {}
            for k, v in ex_clean.items():
                if k == "output" and not inserted:
                    new_ex["prompt"] = prompt_text
                    inserted = True
                new_ex[k] = v

    if not inserted:
        new_ex["prompt"] = prompt_text

    return new_ex

def reinsert_context_and_prompt(chunk_it_wo_context, chunk_en_wo_prompt, base_prompt: str, test: bool):
    """
    chunk_it_wo_context: translated examples WITHOUT 'context' (and typically without 'prompt')
    chunk_en_wo_prompt: EN reference examples WITHOUT 'prompt' (but WITH 'context')
    """
    if len(chunk_it_wo_context) != len(chunk_en_wo_prompt):
        raise ValueError(f"Length mismatch: IT={len(chunk_it_wo_context)} EN={len(chunk_en_wo_prompt)}")

    out = []
    for it_ex, en_ex in zip(chunk_it_wo_context, chunk_en_wo_prompt):
        d = dict(it_ex)

        # Reinsert context from EN (unchanged)
        d["context"] = en_ex.get("context", "")

        # Enforce question_id from EN to avoid any drift
        if "question_id" in en_ex:
            d["question_id"] = en_ex.get("question_id")

        # Build per-example prompt using (IT question) + (EN context)
        question_val = d.get("question", "")
        context_val = d.get("context", "")

        full_prompt = build_full_prompt(base_prompt, question_val, context_val, test=test)
        d2 = insert_prompt_in_order(d, full_prompt, test=test)

        out.append(d2)

    return out

def append_jsonl(path: Path, examples):
    with path.open("a", encoding="utf-8") as f:
        for ex in examples:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

# IMPORTANT:
# - chunk_it_wo_prompt must now be the IT chunk you loaded from file (without context, without prompt)
# - chunk_en_wo_prompt must be the EN reference chunk you already reconstruct in your notebook (without prompt, with context)

chunk_it_full = reinsert_context_and_prompt(chunk_it_wo_prompt, chunk_en_wo_prompt, prompt_it, test=test)
append_jsonl(OUT_IT_JSONL, chunk_it_full)

print(f"Appended {len(chunk_it_full)} examples to: {OUT_IT_JSONL.resolve()}")
print("Mode:", "TEST" if test else "TRAIN")

Appended 6 examples to: /Users/usermastro/Desktop/Primo_Semestre_2526/DNLP/Project/TISER_repo/tools/translation/TISER_it.jsonl
Mode: TEST


# Questa qua sotto va inserita al posto di quella sopra

In [ ]:
from pathlib import Path
import json

OUT_IT_JSONL = Path("TISER_it.jsonl")  # change name if you want

# Set this flag:
test = True  # True = test dataset schema, False = train dataset schema

def build_full_prompt(base_prompt: str, question_val: str, context_val: str, test: bool) -> str:
    """
    Build the per-example prompt.

    - test=True: Question + Temporal context must appear BEFORE '### Answer:'.
      Assumes base_prompt ends with '### Answer:' (or at least contains it).
      We inject Q/ctx right before the last occurrence of '### Answer:'.

    - test=False: Question + Temporal context must be appended at the END of the prompt.
    """
    if test:
        marker = "### Answer:"
        if marker in base_prompt:
            idx = base_prompt.rfind(marker)
            head = base_prompt[:idx].rstrip()
            tail = base_prompt[idx:]  # includes '### Answer:'
            return (
                f"{head}\n\n"
                f"Question: {question_val}\n\n"
                f"Temporal context: {context_val}\n\n"
                f"{tail}"
            )
        else:
            return (
                f"{base_prompt.strip()}\n\n"
                f"Question: {question_val}\n\n"
                f"Temporal context: {context_val}\n\n"
                f"### Answer:"
            )
    else:
        return (
            f"{base_prompt.strip()}\n\n"
            f"Question: {question_val}\n\n"
            f"Temporal context: {context_val}"
        )

def insert_prompt_in_order(ex: dict, prompt_text: str, test: bool) -> dict:
    """
    Insert 'prompt' in the dictionary order depending on dataset type:
    - test=True: after 'question' and before 'answer'
    - test=False: after 'answer' and before 'output'
    Keeps all other keys in their original relative order.
    """
    ex_clean = {k: v for k, v in ex.items() if k != "prompt"}  # remove any existing prompt

    new_ex = {}
    inserted = False

    if test:
        for k, v in ex_clean.items():
            new_ex[k] = v
            if k == "question" and not inserted:
                new_ex["prompt"] = prompt_text
                inserted = True

        if not inserted:
            new_ex = {}
            for k, v in ex_clean.items():
                if k == "answer" and not inserted:
                    new_ex["prompt"] = prompt_text
                    inserted = True
                new_ex[k] = v

    else:
        for k, v in ex_clean.items():
            new_ex[k] = v
            if k == "answer" and not inserted:
                new_ex["prompt"] = prompt_text
                inserted = True

        if not inserted:
            new_ex = {}
            for k, v in ex_clean.items():
                if k == "output" and not inserted:
                    new_ex["prompt"] = prompt_text
                    inserted = True
                new_ex[k] = v

    if not inserted:
        new_ex["prompt"] = prompt_text

    return new_ex

def reinsert_prompt_and_force_context_last(examples, base_prompt: str, test: bool):
    out = []
    for ex in examples:
        d = dict(ex)

        # Grab context (may be missing if you removed it earlier)
        context_val = d.get("context", "")

        # Build prompt using question + context
        question_val = d.get("question", "")
        full_prompt = build_full_prompt(base_prompt, question_val, context_val, test=test)

        # Insert prompt in the right position
        fixed = insert_prompt_in_order(d, full_prompt, test=test)

        # Force 'context' to be the LAST key
        ctx = fixed.pop("context", None)
        if ctx is None:
            ctx = context_val  # fallback if absent
        fixed["context"] = ctx

        out.append(fixed)
    return out

def append_jsonl(path: Path, examples):
    with path.open("a", encoding="utf-8") as f:
        for ex in examples:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")

# Build full chunk with dynamic prompts, forcing context last
chunk_it_full = reinsert_prompt_and_force_context_last(chunk_it_wo_prompt, prompt_it, test=test)

# Append to output jsonl
append_jsonl(OUT_IT_JSONL, chunk_it_full)

print(f"Appended {len(chunk_it_full)} examples to: {OUT_IT_JSONL.resolve()}")
print("Mode:", "TEST" if test else "TRAIN")

---

---

### Correzione e reinserimento del prompt nel dataset TISER (versione italiana)

Questa cella serve a **ripulire e correggere il campo `prompt`** all’interno del file  
`tools/translation/TISER_it.jsonl`, producendo un nuovo dataset coerente con il formato corretto richiesto da TISER.

In particolare, la cella esegue le seguenti operazioni per **ogni oggetto JSON (una riga)** del file:

1. **Carica il prompt corretto**  
   Il testo del prompt viene letto dal file  
   `tools/translation/real_TISER_prompt_it.txt`  
   e utilizzato identicamente per tutti i sample.

2. **Rimuove il campo `prompt` esistente**  
   Qualunque campo `prompt` già presente nell’esempio (errato nel contenuto o nella posizione)
   viene eliminato per evitare duplicazioni o ambiguità.

3. **Reinserisce il prompt nella posizione corretta**  
   Il campo `prompt` viene reinserito **immediatamente prima del campo `output`**,  
   preservando l’ordine degli altri campi del JSON.

4. **Preserva struttura e contenuto del dataset**  
   Nessun altro campo viene modificato:
   - i valori di `question`, `answer`, `output`, `context`, ecc. restano invariati;
   - l’ordine dei campi è mantenuto, salvo l’inserimento corretto del `prompt`.

5. **Scrive un nuovo file di output**  
   Il dataset corretto viene salvato in  
   `tools/translation/TISER_it.fixed.jsonl`,  
   lasciando intatto il file originale per sicurezza.

6. **Controllo di integrità**  
   Se un esempio non contiene il campo `output`, la cella solleva un errore,
   poiché non sarebbe possibile posizionare correttamente il prompt.

Al termine dell’esecuzione, la cella stampa:
- il numero totale di esempi letti;
- il numero di esempi corretti e scritti nel nuovo file.

Questo passaggio garantisce che il dataset italiano TISER rispetti
esattamente il formato previsto dal modello, sia dal punto di vista
semantico sia strutturale.

In [70]:
from pathlib import Path
import json

# Percorsi dei file
IN_PATH = Path("TISER_it.jsonl")
PROMPT_PATH = Path("test_TISER_prompt_it.txt")
OUT_PATH = Path("test_TISER_it.fixed.jsonl")

# Carica il testo base del prompt tradotto
# DEVE contenere "### Answer:" come anchor finale
base_prompt_it = PROMPT_PATH.read_text(encoding="utf-8").strip()

def fix_example_order_and_prompt_test(ex: dict, base_text: str) -> dict:
    """
    TEST dataset:
    - Question e Temporal context vengono inseriti PRIMA di '### Answer:'
    - il campo 'prompt' viene posizionato DOPO 'question' e PRIMA di 'answer'
    """
    question_val = ex.get("question", "")
    context_val = ex.get("context", "")

    marker = "### Answer:"

    if marker in base_text:
        idx = base_text.rfind(marker)
        head = base_text[:idx].rstrip()
        tail = base_text[idx:]  # include ### Answer:
        full_prompt = (
            f"{head}\n\n"
            f"Question: {question_val}\n\n"
            f"Temporal context: {context_val}\n\n"
            f"{tail}"
        )
    else:
        # fallback di sicurezza (non ideale, ma robusto)
        full_prompt = (
            f"{base_text}\n\n"
            f"Question: {question_val}\n\n"
            f"Temporal context: {context_val}\n\n"
            f"### Answer:"
        )

    # Rimuove eventuale prompt pre-esistente
    ex_clean = {k: v for k, v in ex.items() if k != "prompt"}

    # Inserisce prompt dopo 'question' e prima di 'answer'
    new_ex = {}
    inserted = False

    for k, v in ex_clean.items():
        new_ex[k] = v
        if k == "question" and not inserted:
            new_ex["prompt"] = full_prompt
            inserted = True

    # fallback se 'question' manca
    if not inserted:
        new_ex = {}
        for k, v in ex_clean.items():
            if k == "answer" and not inserted:
                new_ex["prompt"] = full_prompt
                inserted = True
            new_ex[k] = v

    if not inserted:
        new_ex["prompt"] = full_prompt

    return new_ex

# Processamento del file
n_total = 0
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with IN_PATH.open("r", encoding="utf-8") as fin, OUT_PATH.open("w", encoding="utf-8") as fout:
    for line_no, line in enumerate(fin, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            ex = json.loads(line)
        except json.JSONDecodeError as e:
            print(f"Errore JSON alla riga {line_no}: {e}")
            continue

        fixed_ex = fix_example_order_and_prompt_test(ex, base_prompt_it)
        fout.write(json.dumps(fixed_ex, ensure_ascii=False) + "\n")
        n_total += 1

print("Elaborazione completata!")
print(f"Letti {n_total} esempi da: {IN_PATH.resolve()}")
print(f"Creato file corretto con prompt dinamici in: {OUT_PATH.resolve()}")

Elaborazione completata!
Letti 2095 esempi da: /Users/usermastro/Desktop/Primo_Semestre_2526/DNLP/Project/TISER_repo/tools/translation/TISER_it.jsonl
Creato file corretto con prompt dinamici in: /Users/usermastro/Desktop/Primo_Semestre_2526/DNLP/Project/TISER_repo/tools/translation/test_TISER_it.fixed.jsonl


## PER TRAIN GIù

In [867]:
from pathlib import Path
import json

# Percorsi dei file
IN_PATH = Path("TISER_it.jsonl")
PROMPT_PATH = Path("real_TISER_prompt_it.txt")
OUT_PATH = Path("tools/translation/TISER_it.fixed.jsonl")

# Carica il testo base del prompt tradotto
# Assicurati che il file .txt finisca dopo </answer> senza "Question:" extra
base_prompt_it = PROMPT_PATH.read_text(encoding="utf-8").strip()

def fix_example_order_and_prompt(ex: dict, base_text: str) -> dict:
    """
    Ricostruisce l'ordine delle chiavi e genera un prompt dinamico
    includendo Question e Temporal context specifici per ogni sample.
    """
    # 1. Recupera i dati specifici del sample
    # Usiamo i nomi delle chiavi originali "question" e "context" dal dataset
    question_val = ex.get("question", "")
    context_val = ex.get("context", "")

    # 2. Costruisce il prompt completo (Base + Question + Context)
    # Segue esattamente il formato del dataset originale inglese
    full_prompt = (
        f"{base_text}\n\n"
        f"        Question: {question_val}\n\n"
        f"        Temporal context: {context_val}"
    )

    # 3. Rimuove eventuali chiavi 'prompt' esistenti per evitare duplicati o posizioni errate
    ex_clean = {k: v for k, v in ex.items() if k != "prompt"}

    if "output" not in ex_clean:
        # Fallback nel caso la struttura sia diversa dal previsto
        ex_clean["prompt"] = full_prompt
        return ex_clean

    # 4. Ricostruisce il dizionario inserendo 'prompt' esattamente prima di 'output'
    new_ex = {}
    for k, v in ex_clean.items():
        if k == "output":
            new_ex["prompt"] = full_prompt
        new_ex[k] = v
        
    return new_ex

# Processamento del file
n_total = 0
n_fixed = 0

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with IN_PATH.open("r", encoding="utf-8") as fin, OUT_PATH.open("w", encoding="utf-8") as fout:
    for line_no, line in enumerate(fin, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            ex = json.loads(line)
        except json.JSONDecodeError as e:
            print(f"Errore JSON alla riga {line_no}: {e}")
            continue

        # Applica la trasformazione
        fixed_ex = fix_example_order_and_prompt(ex, base_prompt_it)

        # Scrive la riga nel nuovo file mantenendo i caratteri UTF-8 corretti
        fout.write(json.dumps(fixed_ex, ensure_ascii=False) + "\n")
        n_total += 1
        n_fixed += 1

print(f"Elaborazione completata!")
print(f"Letti {n_total} esempi da: {IN_PATH.resolve()}")
print(f"Creato file corretto con prompt dinamici in: {OUT_PATH.resolve()}")

Elaborazione completata!
Letti 5370 esempi da: /Users/usermastro/Desktop/Primo_Semestre_2526/DNLP/Project/TISER_repo/tools/translation/TISER_it.jsonl
Creato file corretto con prompt dinamici in: /Users/usermastro/Desktop/Primo_Semestre_2526/DNLP/Project/TISER_repo/tools/translation/tools/translation/TISER_it.fixed.jsonl


---

---

## Notes on Translation

When translating a chunk:

- **Preserve the JSON structure exactly**
- Do NOT rename keys
- Do NOT remove or add fields
- Translate only natural language content:
  - question
  - context
  - reasoning
  - timeline
  - reflection
  - answer
- Keep all special tags unchanged:
  `<reasoning>`, `<timeline>`, `<reflection>`, `<answer>`

Once translated, the chunk can be appended to the target dataset
using the companion script.